# BirdCLEF 2026: Validation + Inference + Submission

This notebook implements the final two pipeline steps in one self-contained file:

6. **Validation**: grouped validation by soundscape file or site, measured with macro ROC-AUC that skips classes with no positive/negative labels.
7. **Inference**: split hidden test soundscapes into 5-second windows, predict 234 probabilities, and create `submission.csv`.

There are no helper scripts. The model class, audio functions, validation metric, inference code, and submission writer are all inside this notebook.

## 0. Imports and Hardware Check

In [10]:
from pathlib import Path
import gc
import json
import math
import os
import platform
import random
import re
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyTorch is required. Select the project kernel and install torch first.") from exc

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)
if torch.cuda.is_available():
    print("CUDA version used by PyTorch:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

try:
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
    if result.returncode == 0:
        print("\n".join(result.stdout.splitlines()[:14]))
except Exception:
    pass

Python: 3.11.4
Platform: Windows-10-10.0.19045-SP0
Torch: 2.11.0+cu128
CUDA available: True
Selected device: cuda
CUDA version used by PyTorch: 12.8
GPU: NVIDIA GeForce RTX 3060
VRAM GB: 12.0
Tue Jun  2 14:36:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.49                 Driver Version: 596.49         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060      WDDM  |   00000000:01:00.0  On |                  N/A |
|  0%   55C    P8             24W /  170

## 1. Configuration

In [11]:
PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / "birdclef_work"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"

def looks_like_birdclef_dataset(path):
    required = ["taxonomy.csv", "sample_submission.csv", "test_soundscapes"]
    return path.exists() and path.is_dir() and all((path / item).exists() for item in required)

def find_dataset_dir():
    candidates = []
    env_dir = os.environ.get("BIRDCLEF_DATA_DIR")
    if env_dir:
        candidates.append(Path(env_dir))
    for base in [PROJECT_DIR, PROJECT_DIR.parent, Path("/kaggle/input")]:
        candidates.extend([base / "birdclef-2026", base / "birdcle-2026"])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.iterdir() if p.is_dir()])
    for candidate in candidates:
        if looks_like_birdclef_dataset(candidate):
            return candidate.resolve()
    checked = "\n".join(str(c) for c in candidates)
    raise FileNotFoundError(f"Could not find the BirdCLEF dataset folder. Checked:\n{checked}")

DATA_DIR = find_dataset_dir()
TEST_DIR = DATA_DIR / "test_soundscapes"

TARGET_SR = 32_000
CLIP_SECONDS = 5
CLIP_SAMPLES = TARGET_SR * CLIP_SECONDS
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 20
FMAX = TARGET_SR // 2

BATCH_SIZE = 64 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 0
VAL_FRACTION = 0.20

# Options: "existing", "file", "site".
# "existing" uses notebook 02's split if available; that split is grouped by file.
VALIDATION_SPLIT_MODE = "existing"

# Checkpoint priority: Stage 3 best first, then Stage 2 best, then Stage 2 last.
# On Kaggle, attach/upload your trained checkpoint as an input dataset. This notebook also searches /kaggle/input.
CHECKPOINT_NAMES = [
    "stage3_pseudolabel_best.pt",
    "stage2_soundscape_best.pt",
    "stage2_soundscape_last.pt",
]

def build_checkpoint_candidates():
    candidates = []
    for name in CHECKPOINT_NAMES:
        candidates.append(CHECKPOINT_DIR / name)
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        found = list(kaggle_input.rglob("*.pt"))
        for name in CHECKPOINT_NAMES:
            candidates.extend([p for p in found if p.name == name])
        candidates.extend(found)
    unique = []
    seen = set()
    for path in candidates:
        path = Path(path)
        key = str(path)
        if key not in seen:
            unique.append(path)
            seen.add(key)
    return unique

CHECKPOINT_CANDIDATES = build_checkpoint_candidates()

VALIDATION_PREDICTIONS_PATH = WORK_DIR / "validation_predictions.csv"
VALIDATION_METRICS_PATH = WORK_DIR / "validation_metrics.json"
SUBMISSION_PATH = PROJECT_DIR / "submission.csv"

print("Dataset:", DATA_DIR)
print("Test soundscapes:", TEST_DIR)
print("Work dir:", WORK_DIR)
print("Batch size:", BATCH_SIZE)
print("Validation split mode:", VALIDATION_SPLIT_MODE)

Dataset: D:\BirdCLEF+\birdclef-2026
Test soundscapes: D:\BirdCLEF+\birdclef-2026\test_soundscapes
Work dir: d:\BirdCLEF+\birdclef_work
Batch size: 64
Validation split mode: existing


## 2. Load Metadata

In [12]:
label_map_path = WORK_DIR / "label_map.csv"
soundscape_windows_path = WORK_DIR / "train_soundscape_windows.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

for path in [label_map_path, soundscape_windows_path, sample_submission_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

label_map = pd.read_csv(label_map_path, dtype={"primary_label": str})
target_labels = label_map["primary_label"].astype(str).tolist()
num_classes = len(target_labels)

sample_submission = pd.read_csv(sample_submission_path)
sample_target_labels = [c for c in sample_submission.columns if c != "row_id"]
if sample_target_labels != target_labels:
    raise ValueError("Target label order in label_map.csv does not match sample_submission.csv.")

soundscape_df = pd.read_csv(soundscape_windows_path, dtype={"filename": str})
soundscape_df["audio_path"] = soundscape_df["audio_path"].astype(str)
soundscape_df["start_seconds"] = soundscape_df["start_seconds"].astype(float)
soundscape_df[target_labels] = soundscape_df[target_labels].astype(np.float32)

def extract_site(filename):
    match = re.search(r"_S(\d+)_", str(filename))
    return f"S{match.group(1)}" if match else "unknown"

soundscape_df["site"] = soundscape_df["filename"].apply(extract_site)

missing_audio = [p for p in soundscape_df["audio_path"].unique() if not Path(p).exists()]
if missing_audio:
    raise FileNotFoundError(f"Some validation audio files are missing. First missing path: {missing_audio[0]}")

print("Targets:", num_classes)
print("Labeled soundscape windows:", len(soundscape_df))
print("Labeled soundscape files:", soundscape_df["filename"].nunique())
print("Labeled sites:", sorted(soundscape_df["site"].unique()))
print("Sample submission rows:", len(sample_submission))
soundscape_df.head()

Targets: 234
Labeled soundscape windows: 1478
Labeled soundscape files: 66
Labeled sites: ['S03', 'S08', 'S09', 'S13', 'S15', 'S18', 'S19', 'S22', 'S23']
Sample submission rows: 3


,filename,audio_path,file_exists,start,end,start_seconds,end_seconds,duration_seconds,labels_string,positive_count,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,47158son10,...,schpar1,scther1,shcfly1,shshaw,shtnig1,sibtan2,smbani,smbtin1,sobcac1,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1,site
0,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:00,00:00:05,0.0,5,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,S22
1,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:05,00:00:10,5.0,10,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,S22
2,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:10,00:00:15,10.0,15,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,S22
3,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:15,00:00:20,15.0,20,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,S22
4,BC2026_Train_0039_S22_20211231_201500.ogg,D:\BirdCLEF+\birdclef-2026\train_soundscapes\BC2026_Train_0039_S22_20211231_201500.ogg,True,00:00:20,00:00:25,20.0,25,5,22961;23158;24321;517063;65380,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,S22


## 3. Audio Backend and Preprocessing

In [13]:
AUDIO_BACKEND = None

try:
    import librosa
    AUDIO_BACKEND = "librosa"
    print("Audio backend: librosa", librosa.__version__)
except Exception as librosa_exc:
    librosa = None
    try:
        import torchaudio
        AUDIO_BACKEND = "torchaudio"
        print("Audio backend: torchaudio", torchaudio.__version__)
    except Exception as torchaudio_exc:
        torchaudio = None
        raise ModuleNotFoundError(
            "This notebook needs either librosa+soundfile or torchaudio to read .ogg files."
        ) from torchaudio_exc

def pad_or_trim(y, target_len=CLIP_SAMPLES):
    y = np.asarray(y, dtype=np.float32)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    elif len(y) > target_len:
        y = y[:target_len]
    return y

def load_audio(path, offset_seconds=0.0, duration_seconds=CLIP_SECONDS, target_sr=TARGET_SR):
    path = Path(path)
    if AUDIO_BACKEND == "librosa":
        y, _ = librosa.load(path, sr=target_sr, mono=True, offset=float(offset_seconds), duration=float(duration_seconds))
        return pad_or_trim(y, int(target_sr * duration_seconds))

    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    start = int(round(float(offset_seconds) * sr))
    end = start + int(round(float(duration_seconds) * sr))
    waveform = waveform[:, start:end]
    if sr != target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, target_sr)
    return pad_or_trim(waveform.squeeze(0).numpy(), int(target_sr * duration_seconds))

def get_audio_duration(path):
    path = Path(path)
    if AUDIO_BACKEND == "librosa":
        return float(librosa.get_duration(path=str(path)))
    info = torchaudio.info(str(path))
    return float(info.num_frames / info.sample_rate)

if AUDIO_BACKEND == "torchaudio":
    MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
        sample_rate=TARGET_SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        f_min=FMIN,
        f_max=FMAX,
        power=2.0,
    )
    DB_TRANSFORM = torchaudio.transforms.AmplitudeToDB(stype="power")

def waveform_to_logmel(y):
    y = np.asarray(y, dtype=np.float32)
    if AUDIO_BACKEND == "librosa":
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=TARGET_SR,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS,
            fmin=FMIN,
            fmax=FMAX,
            power=2.0,
        )
        logmel = librosa.power_to_db(mel, ref=np.max)
    else:
        waveform = torch.tensor(y, dtype=torch.float32).unsqueeze(0)
        logmel = DB_TRANSFORM(MEL_TRANSFORM(waveform)).squeeze(0).numpy()
    mean = float(logmel.mean())
    std = float(logmel.std())
    return ((logmel - mean) / (std + 1e-6)).astype(np.float32)

def seconds_to_hhmmss(seconds):
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

Audio backend: librosa 0.11.0


## 4. Model Definition and Checkpoint Loading

This model class must match Notebook 02, otherwise the checkpoint cannot load.

In [14]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class BirdCLEFSmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
            nn.MaxPool2d(2),
            ConvBlock(64, 128),
            nn.MaxPool2d(2),
            ConvBlock(128, 256),
            nn.MaxPool2d(2),
            ConvBlock(256, 384),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Linear(384, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.classifier(x)

def choose_checkpoint(candidates):
    for path in candidates:
        if Path(path).exists():
            return Path(path)
    raise FileNotFoundError(
        "No trained checkpoint was found. Run Notebook 02 first. Checked:\n" + "\n".join(str(p) for p in candidates)
    )

def torch_load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

checkpoint_path = choose_checkpoint(CHECKPOINT_CANDIDATES)
checkpoint = torch_load_checkpoint(checkpoint_path)

checkpoint_labels = checkpoint.get("target_labels") if isinstance(checkpoint, dict) else None
if checkpoint_labels is not None and list(checkpoint_labels) != target_labels:
    raise ValueError("Checkpoint target label order does not match label_map.csv.")

model = BirdCLEFSmallCNN(num_classes=num_classes).to(DEVICE)
state = checkpoint.get("model_state_dict", checkpoint) if isinstance(checkpoint, dict) else checkpoint
state = {k.replace("module.", ""): v for k, v in state.items()}
missing, unexpected = model.load_state_dict(state, strict=False)
if missing or unexpected:
    print("Missing keys:", missing)
    print("Unexpected keys:", unexpected)
    if len(unexpected) > 0:
        raise RuntimeError("Checkpoint has unexpected model keys. Check model architecture compatibility.")

model.eval()
print("Loaded checkpoint:", checkpoint_path)
if isinstance(checkpoint, dict):
    print("Checkpoint stage:", checkpoint.get("stage_name"))
    print("Checkpoint epoch:", checkpoint.get("epoch"))
    print("Checkpoint best AUC:", checkpoint.get("best_auc"))

Loaded checkpoint: d:\BirdCLEF+\birdclef_work\checkpoints\stage2_soundscape_best.pt
Checkpoint stage: stage2_soundscape
Checkpoint epoch: 10
Checkpoint best AUC: 0.8967637848241443


## 5. Validation Split and Macro ROC-AUC

In [15]:
def create_grouped_split(df, mode="file", val_fraction=VAL_FRACTION):
    if mode not in {"file", "site"}:
        raise ValueError("mode must be 'file' or 'site'")
    group_col = "filename" if mode == "file" else "site"
    groups = sorted(df[group_col].unique())
    rng = np.random.default_rng(SEED)
    shuffled = groups.copy()
    rng.shuffle(shuffled)
    val_count = max(1, int(round(len(shuffled) * val_fraction)))
    val_groups = set(shuffled[:val_count])
    split = np.where(df[group_col].isin(val_groups), "val", "train")
    return pd.Series(split, index=df.index), group_col, val_groups

def apply_validation_split(df):
    df = df.copy()
    split_file = WORK_DIR / "stage2_soundscape_split.csv"
    if VALIDATION_SPLIT_MODE == "existing" and split_file.exists():
        split_df = pd.read_csv(split_file, dtype={"filename": str})
        split_df = split_df[["filename", "start", "end", "split"]].drop_duplicates()
        merged = df.merge(split_df, on=["filename", "start", "end"], how="left")
        if merged["split"].isna().any():
            print("Existing split did not cover every row. Falling back to grouped file split.")
            merged["split"], group_col, val_groups = create_grouped_split(merged, mode="file")
            split_used = "file_fallback"
        else:
            group_col = "filename"
            val_groups = set(merged.loc[merged["split"].eq("val"), group_col].unique())
            split_used = "existing_stage2_file_split"
        return merged, split_used, group_col, val_groups

    mode = "site" if VALIDATION_SPLIT_MODE == "site" else "file"
    df["split"], group_col, val_groups = create_grouped_split(df, mode=mode)
    return df, f"fresh_{mode}_split", group_col, val_groups

def average_ranks(values):
    values = np.asarray(values)
    order = np.argsort(values, kind="mergesort")
    sorted_values = values[order]
    ranks = np.empty(len(values), dtype=np.float64)
    i = 0
    while i < len(values):
        j = i + 1
        while j < len(values) and sorted_values[j] == sorted_values[i]:
            j += 1
        avg_rank = (i + 1 + j) / 2.0
        ranks[order[i:j]] = avg_rank
        i = j
    return ranks

def macro_auc_skip_empty(y_true, y_score):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_score = np.asarray(y_score, dtype=np.float32)
    aucs = []
    used_indices = []
    for c in range(y_true.shape[1]):
        yt = y_true[:, c]
        ys = y_score[:, c]
        n_pos = int(yt.sum())
        n_neg = int(len(yt) - n_pos)
        if n_pos == 0 or n_neg == 0:
            continue
        ranks = average_ranks(ys)
        pos_rank_sum = ranks[yt == 1].sum()
        auc = (pos_rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
        aucs.append(float(auc))
        used_indices.append(c)
    if not aucs:
        return float("nan"), 0, pd.DataFrame()
    per_class = pd.DataFrame({
        "primary_label": [target_labels[i] for i in used_indices],
        "auc": aucs,
        "positive_count": [int(y_true[:, i].sum()) for i in used_indices],
    })
    per_class = per_class.merge(label_map[["primary_label", "common_name", "class_name"]], on="primary_label", how="left")
    return float(np.mean(aucs)), len(aucs), per_class.sort_values("auc")

split_df, split_used, group_col, val_groups = apply_validation_split(soundscape_df)
train_split_df = split_df[split_df["split"].eq("train")].reset_index(drop=True)
val_df = split_df[split_df["split"].eq("val")].reset_index(drop=True)

if len(val_df) == 0:
    raise ValueError("Validation split is empty.")

print("Split used:", split_used)
print("Grouped by:", group_col)
print("Train rows/files/sites:", len(train_split_df), train_split_df["filename"].nunique(), train_split_df["site"].nunique())
print("Val rows/files/sites:", len(val_df), val_df["filename"].nunique(), val_df["site"].nunique())
print("Validation positive classes:", int((val_df[target_labels].sum(axis=0) > 0).sum()))
print("Validation groups:", sorted(list(val_groups))[:20], "..." if len(val_groups) > 20 else "")

Split used: existing_stage2_file_split
Grouped by: filename
Train rows/files/sites: 1180 53 9
Val rows/files/sites: 298 13 4
Validation positive classes: 33
Validation groups: ['BC2026_Train_0008_S09_20250831_000000.ogg', 'BC2026_Train_0018_S22_20211028_234500.ogg', 'BC2026_Train_0019_S22_20211104_200000.ogg', 'BC2026_Train_0022_S22_20211114_014500.ogg', 'BC2026_Train_0030_S22_20211212_224500.ogg', 'BC2026_Train_0033_S22_20211216_200000.ogg', 'BC2026_Train_0043_S22_20220112_040000.ogg', 'BC2026_Train_0045_S22_20220125_211500.ogg', 'BC2026_Train_0051_S22_20220208_231500.ogg', 'BC2026_Train_0052_S22_20220210_210000.ogg', 'BC2026_Train_0058_S15_20250617_060100.ogg', 'BC2026_Train_0059_S15_20250617_060200.ogg', 'BC2026_Train_0066_S23_20241124_044002.ogg'] 


## 6. Run Validation

In [16]:
class LabeledWindowDataset(Dataset):
    def __init__(self, df, target_labels):
        self.df = df.reset_index(drop=True)
        self.target_labels = target_labels
        self.paths = self.df["audio_path"].astype(str).values
        self.starts = self.df["start_seconds"].astype(float).values
        self.targets = self.df[target_labels].values.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        waveform = load_audio(self.paths[idx], offset_seconds=self.starts[idx], duration_seconds=CLIP_SECONDS)
        logmel = waveform_to_logmel(waveform)
        x = torch.from_numpy(logmel).unsqueeze(0)
        y = torch.from_numpy(self.targets[idx])
        return x, y

@torch.no_grad()
def predict_labeled_windows(model, df):
    ds = LabeledWindowDataset(df, target_labels)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    all_true = []
    all_score = []
    model.eval()
    for x, y in tqdm(loader, desc="validation"):
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_score.append(probs)
        all_true.append(y.numpy())
    return np.concatenate(all_true, axis=0), np.concatenate(all_score, axis=0)

start_time = time.time()
y_true, y_score = predict_labeled_windows(model, val_df)
macro_auc, used_classes, per_class_auc = macro_auc_skip_empty(y_true, y_score)
elapsed = time.time() - start_time

validation_predictions = val_df[["filename", "site", "start", "end", "start_seconds", "labels_string"]].copy()
for i, label in enumerate(target_labels):
    validation_predictions[label] = y_score[:, i]
validation_predictions.to_csv(VALIDATION_PREDICTIONS_PATH, index=False)

metrics = {
    "checkpoint_path": str(checkpoint_path),
    "split_used": split_used,
    "group_col": group_col,
    "val_rows": int(len(val_df)),
    "val_files": int(val_df["filename"].nunique()),
    "val_sites": int(val_df["site"].nunique()),
    "macro_auc_skip_empty": macro_auc,
    "auc_classes_used": int(used_classes),
    "seconds": elapsed,
}
VALIDATION_METRICS_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
per_class_auc.to_csv(WORK_DIR / "validation_per_class_auc.csv", index=False)

print(json.dumps(metrics, indent=2))
print("Saved validation predictions:", VALIDATION_PREDICTIONS_PATH)
print("Saved validation metrics:", VALIDATION_METRICS_PATH)
print("Saved per-class AUC:", WORK_DIR / "validation_per_class_auc.csv")
per_class_auc.head(15)

{
  "checkpoint_path": "d:\\BirdCLEF+\\birdclef_work\\checkpoints\\stage2_soundscape_best.pt",
  "split_used": "existing_stage2_file_split",
  "group_col": "filename",
  "val_rows": 298,
  "val_files": 13,
  "val_sites": 4,
  "macro_auc_skip_empty": 0.8967637848241443,
  "auc_classes_used": 33,
  "seconds": 4.945472002029419
}
Saved validation predictions: d:\BirdCLEF+\birdclef_work\validation_predictions.csv
Saved validation metrics: d:\BirdCLEF+\birdclef_work\validation_metrics.json
Saved per-class AUC: d:\BirdCLEF+\birdclef_work\validation_per_class_auc.csv


,primary_label,auc,positive_count,common_name,class_name
28,rufhor2,0.463793,8,Rufous Hornero,Aves
18,bunibi1,0.602041,4,Buff-necked Ibis,Aves
9,47158son08,0.670833,10,Insect sonotype08,Insecta
1,22967,0.697401,96,Marbled White-lipped Frog,Amphibia
17,bufpar,0.809524,4,Turquoise-fronted Amazon,Aves
4,24279,0.854194,50,Lesser Snouted Tree Frog,Amphibia
24,magant1,0.863946,4,Mato Grosso Antbird,Aves
13,517063,0.874017,106,Southern Orange-legged Leaf Frog,Amphibia
31,undtin1,0.889328,22,Undulated Tinamou,Aves
19,chacha1,0.893156,66,Chaco Chachalaca,Aves


## 7. Build Test Manifest

In the local public dataset, `test_soundscapes/` only contains `readme.txt`. During the Kaggle hidden rerun, this folder is populated with `.ogg` files. This cell handles both cases.

In [17]:
def parse_row_id(row_id):
    stem, end_text = str(row_id).rsplit("_", 1)
    return stem, int(end_text)

def build_manifest_from_sample_submission(sample_df, test_dir):
    rows = []
    missing = []
    for row_id in sample_df["row_id"].astype(str):
        stem, end_seconds = parse_row_id(row_id)
        path = test_dir / f"{stem}.ogg"
        if not path.exists():
            missing.append(str(path))
            continue
        rows.append({
            "row_id": row_id,
            "filename": path.name,
            "audio_path": str(path),
            "start_seconds": float(end_seconds - CLIP_SECONDS),
            "end_seconds": float(end_seconds),
        })
    if missing:
        return pd.DataFrame(), missing
    return pd.DataFrame(rows), []

def build_manifest_from_test_files(test_dir):
    rows = []
    test_files = sorted(test_dir.glob("*.ogg"))
    for path in tqdm(test_files, desc="scan test files"):
        duration = get_audio_duration(path)
        full_windows = int(duration // CLIP_SECONDS)
        remainder = duration - full_windows * CLIP_SECONDS
        n_windows = full_windows + (1 if remainder > 1.0 else 0)
        for w in range(n_windows):
            start_seconds = w * CLIP_SECONDS
            end_seconds = start_seconds + CLIP_SECONDS
            rows.append({
                "row_id": f"{path.stem}_{int(end_seconds)}",
                "filename": path.name,
                "audio_path": str(path),
                "start_seconds": float(start_seconds),
                "end_seconds": float(end_seconds),
            })
    return pd.DataFrame(rows)

test_files = sorted(TEST_DIR.glob("*.ogg"))
print("Test .ogg files found:", len(test_files))

if len(test_files) == 0:
    test_manifest = pd.DataFrame()
    print("No local test audio found. This is normal outside the Kaggle hidden rerun.")
else:
    test_manifest, missing_from_sample = build_manifest_from_sample_submission(sample_submission, TEST_DIR)
    if len(test_manifest) > 0:
        print("Using sample_submission row order for test manifest.")
    else:
        print("sample_submission rows did not match local test files. Building manifest from audio durations.")
        if missing_from_sample:
            print("First missing sample path:", missing_from_sample[0])
        test_manifest = build_manifest_from_test_files(TEST_DIR)

print("Test windows:", len(test_manifest))
test_manifest.head()

Test .ogg files found: 0
No local test audio found. This is normal outside the Kaggle hidden rerun.
Test windows: 0


""


## 8. Run Test Inference and Write `submission.csv`

In [18]:
class TestWindowDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.paths = self.df["audio_path"].astype(str).values
        self.starts = self.df["start_seconds"].astype(float).values
        self.row_ids = self.df["row_id"].astype(str).values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        waveform = load_audio(self.paths[idx], offset_seconds=self.starts[idx], duration_seconds=CLIP_SECONDS)
        logmel = waveform_to_logmel(waveform)
        x = torch.from_numpy(logmel).unsqueeze(0)
        return x, self.row_ids[idx]

@torch.no_grad()
def predict_test_windows(model, df):
    ds = TestWindowDataset(df)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
    row_ids = []
    scores = []
    model.eval()
    for x, batch_row_ids in tqdm(loader, desc="test inference"):
        x = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        scores.append(probs)
        row_ids.extend(list(batch_row_ids))
    return row_ids, np.concatenate(scores, axis=0)

if len(test_manifest) == 0:
    # Local dry run fallback: keeps the notebook executable when hidden test audio is not present.
    submission = sample_submission.copy()
    submission.to_csv(SUBMISSION_PATH, index=False)
    print("No test audio found. Wrote sample_submission copy for local dry run:", SUBMISSION_PATH)
else:
    start_time = time.time()
    row_ids, test_scores = predict_test_windows(model, test_manifest)
    test_scores = np.clip(test_scores, 0.0, 1.0)
    submission = pd.DataFrame(test_scores, columns=target_labels)
    submission.insert(0, "row_id", row_ids)
    submission = submission[["row_id"] + target_labels]
    submission.to_csv(SUBMISSION_PATH, index=False)
    print("Inference seconds:", round(time.time() - start_time, 2))
    print("Saved submission:", SUBMISSION_PATH)
    print("Submission shape:", submission.shape)

submission.head()

No test audio found. Wrote sample_submission copy for local dry run: d:\BirdCLEF+\submission.csv


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,22967,22973,22983,22985,23150,23154,23158,23176,23724,24279,24285,24287,24321,244024,25073,25092,25214,326272,41970,43435,47144,47158son01,47158son02,47158son03,47158son04,47158son05,47158son06,47158son07,47158son08,47158son09,47158son10,47158son11,47158son12,47158son13,47158son14,47158son15,47158son16,47158son17,47158son18,47158son19,...,scadov1,schpar1,scther1,shcfly1,shshaw,shtnig1,sibtan2,smbani,smbtin1,sobcac1,sobtyr1,socfly1,sofspi1,souant1,soulap1,souscr1,spbant3,spispi1,sptnig1,squcuc1,stbwoo2,strcuc1,strher2,strowl1,swthum1,swtman1,tattin1,thlwre1,toctou1,trokin,trsowl,undtin1,varant1,watjac1,wesfie1,wfwduc1,whbant2,whbwar2,whiwoo1,whlspi1,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


## Outputs

Validation outputs:

- `birdclef_work/validation_predictions.csv`
- `birdclef_work/validation_metrics.json`
- `birdclef_work/validation_per_class_auc.csv`

Inference output:

- `submission.csv`

On the local public dataset, `submission.csv` may simply be a dry-run copy of `sample_submission.csv` because the public `test_soundscapes/` folder is empty. On Kaggle's hidden rerun, this notebook will process the hidden `.ogg` files and write model predictions.